## Install Dependency

In [1]:
import os
from dotenv import load_dotenv
from typing import Any, Dict, List
from pydantic import PrivateAttr
import pandas as pd
import numpy as np

from distilabel.llms import OpenAILLM
from distilabel.pipeline import Pipeline
from distilabel.steps import (
    LoadDataFromDicts,
    MergeColumns,
    FormatTextGenerationDPO,
    PreferenceToArgilla
)
from distilabel.steps.tasks import TextGeneration, UltraFeedback
from distilabel.typing import ChatType


/tmp/ipykernel_25169/2111529966.py:8: DeprecationWarning: Importing from 'distilabel.llms' is deprecated and will be removed in a version 1.7.0. Import from 'distilabel.models' instead.
  from distilabel.llms import OpenAILLM
/data/ephemeral/pro-nlp-finalproject-nlp-13/.venv/lib/python3.10/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'default' attribute with value None was provided to the `Field()` function, which has no effect in the context it was used. 'default' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/data/ephemeral/pro-nlp-finalproject-nlp-13/.venv/lib/python3.10/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'default' at

In [2]:
load_dotenv()

api_key = os.getenv('UPSTAGE_API_KEY')

## Define the pipeline

In [3]:
df_aihub = pd.read_json('/data/ephemeral/pro-nlp-finalproject-nlp-13/data/persona_data/AiHub_100sample.json', lines=True)

df_aihub.rename(columns={"req": "instruction"}, inplace=True)

data_dicts = df_aihub.to_dict(orient="records")

In [4]:
SNS_CONVERSATION_CRITERIA = {
    "naturalness": {
        "description": "대화가 자연스럽고 실제 SNS에서 사용되는 표현과 어투를 사용하는가",
        "weight": 0.25,
        "rubric": """
        5: 매우 자연스러운 SNS 대화체, 이모티콘/줄임말 적절히 사용
        4: 자연스러운 대화체이나 약간의 어색함
        3: 보통 수준, SNS 특성이 일부 반영됨
        2: 딱딱하거나 형식적인 표현 다수
        1: SNS 대화체와 거리가 먼 격식체
        """
    },
    "contextual_relevance": {
        "description": "이전 대화 맥락을 잘 이해하고 적절하게 응답하는가",
        "weight": 0.25,
        "rubric": """
        5: 맥락을 완벽히 이해하고 자연스럽게 이어감
        4: 맥락을 잘 이해하나 일부 어색한 부분
        3: 맥락을 대체로 이해하나 연결이 약함
        2: 맥락 이해가 부족하거나 동떨어진 응답
        1: 맥락을 완전히 무시한 응답
        """
    },
    "engagement": {
        "description": "대화 상대의 참여를 유도하고 흥미를 유지하는가",
        "weight": 0.20,
        "rubric": """
        5: 공감, 질문, 유머 등으로 활발한 대화 유도
        4: 대화를 이어가기 좋은 응답
        3: 무난하지만 특별히 흥미롭지 않음
        2: 대화를 끊을 수 있는 단답형 응답
        1: 대화 참여 의지가 전혀 느껴지지 않음
        """
    },
    "emotional_appropriateness": {
        "description": "상황과 감정에 적절한 톤과 공감을 표현하는가",
        "weight": 0.15,
        "rubric": """
        5: 상황에 완벽히 맞는 감정 표현과 공감
        4: 적절한 감정 표현
        3: 보통 수준의 감정 반영
        2: 부적절하거나 어긋난 감정 표현
        1: 감정 무시 또는 완전히 부적절한 반응
        """
    },
    "conciseness": {
        "description": "SNS 특성에 맞게 간결하면서도 충분한 정보를 전달하는가",
        "weight": 0.15,
        "rubric": """
        5: 간결하면서 핵심을 정확히 전달
        4: 적절한 길이로 정보 전달
        3: 약간 장황하거나 짧지만 이해 가능
        2: 너무 길거나 짧아서 의미 전달 부족
        1: 극단적으로 부적절한 길이
        """
    }
}

In [5]:
class SNSConversationFeedback(UltraFeedback):
    """SNS 대화체에 특화된 평가 태스크"""

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.aspect = "overall-rating"

    _system_prompt: str = PrivateAttr(
        default="""당신은 실제 SNS 대화체(카카오톡, DM 등)를 판별하는 전문 평가가입니다.

당신의 출력은 오직 아래에 정의된 출력 문법만 허용됩니다.
이를 벗어나는 모든 출력은 파싱 오류로 처리됩니다.

[출력 제약 – 절대 위반 금지]
- 출력은 정확히 4줄이어야 합니다.
- 각 응답은 다음 두 줄로만 구성됩니다:
  1) Rating: 숫자
  2) Rationale: 문장
- 두 응답 묶음 사이에는 정확히 빈 줄 한 줄만 허용됩니다.
- 마지막 Rationale 줄 뒤에는 줄바꿈이나 공백 없이 즉시 종료해야 합니다.
- Rating에는 오직 1~5 사이의 정수만 허용됩니다.
- Rationale은 반드시 한 문장이어야 합니다.
- Rating, Rationale 외의 어떤 텍스트도 출력해서는 안 됩니다.

[금지 사항]
- 추가 설명, 요약, 평가 근거, 분석
- <think> 태그 또는 내부 사고 노출
- 구분선(---), 제목, 번호, 마크다운 기호
- 출력 전후의 공백 줄 또는 문장

[출력 문법]
Rating: [1-5]
Rationale: [한 문장]

Rating: [1-5]
Rationale: [한 문장]
"""
    )

    def format_input(self, input: Dict[str, Any]) -> ChatType:
        """SNS 대화체 판별 및 품질 평가 전용 프롬프트"""

        system_content = self._system_prompt

        # 1. 평가 목적
        user_content = (
            "아래 응답들이 실제 SNS 대화체인지 평가하십시오.\n"
            "평가는 자연스러움, 맥락 이해, 참여 유도, 감정 적절성, 간결성을 종합하여 수행하십시오.\n\n"
        )

        # 2. 다차원 평가 기준
        user_content += (
            "[평가 기준]\n"
            "- 자연스러움: 구어체, 이모티콘, 줄임말 사용 여부\n"
            "- 맥락 적합성: 이전 대화를 이해하고 이어지는가\n"
            "- 참여 유도: 공감, 질문 등 대화 지속 요소\n"
            "- 감정 적절성: 상황에 맞는 톤과 공감\n"
            "- 간결성: SNS 특성에 맞는 길이\n\n"
            "주의: 설명조, 보고서체, 비즈니스 이메일 말투는 낮은 점수를 부여하십시오.\n\n"
        )

        # 3. 대화 맥락
        user_content += f"[지시 사항]\n{input['instruction']}\n\n"

        # 4. 응답 나열
        for i, gen in enumerate(input["generations"], 1):
            user_content += f"[응답 {i}]\n{gen}\n\n"

        # 5. 형식 리마인드 (최소화)
        user_content += "출력은 system에 정의된 출력 문법만 따르십시오."

        return [
            {"role": "system", "content": system_content},
            {"role": "user", "content": user_content},
        ]


In [6]:
with Pipeline(name="sns-conversation-dpo-pipeline") as pipeline:
        load_data = LoadDataFromDicts(
            data=data_dicts,
            batch_size=5
        )

        generate_responses = [
            TextGeneration(
                llm=OpenAILLM(
                    model="solar-mini",
                    base_url="https://api.upstage.ai/v1/solar",
                    api_key="up_KyCFVhEHCYkyBTmQOxujiKk55YVAL",
                    max_retries=10,
                ),
                system_prompt="""당신은 문서체로만 텍스트를 생성해야합니다. 주 목표는
                SNS 대화체와 정반대의 성질을 가지는 말투를 가져야 합니다.""",
            )

        ]

        combine_responses = MergeColumns(
            columns=["res", "generation"],
            output_column="generations"
        )
        

        evaluate_responses = SNSConversationFeedback(
            aspect="overall-rating",
            llm=OpenAILLM(
                model="solar-pro2",
                base_url="https://api.upstage.ai/v1/solar",
                api_key=api_key,
                max_retries=10,
            ),
        )

        format_dpo = FormatTextGenerationDPO()

        load_data >> generate_responses >> combine_responses >> evaluate_responses >> format_dpo



In [7]:
distiset = pipeline.run(use_cache=False)

[02/02/26 08:40:07] INFO     ['distilabel.pipeline'] 📝 Pipeline data will be written to               ]8;id=522703;file:///data/ephemeral/pro-nlp-finalproject-nlp-13/.venv/lib/python3.10/site-packages/distilabel/pipeline/base.py\base.py]8;;\:]8;id=346634;file:///data/ephemeral/pro-nlp-finalproject-nlp-13/.venv/lib/python3.10/site-packages/distilabel/pipeline/base.py#1015\1015]8;;\
                             '/data/ephemeral/home/.cache/distilabel/pipelines/sns-conversation-dpo-pi             
                             peline/9f7a1bca6d1f9f54278bc432001760fc4d085be0/executions/c5f8a475f9382b             
                             a01cd87de786ed8e07db33d51a/data/steps_outputs'                                        

                    INFO     ['distilabel.pipeline'] ⌛ The steps of the pipeline will be loaded in    ]8;id=357072;file:///data/ephemeral/pro-nlp-finalproject-nlp-13/.venv/lib/python3.10/site-packages/distilabel/pipeline/base.py\base.py]8;;\:]8;id=157583;file:///data/ephemeral/pro-nlp-finalproject-nlp-13/.venv/lib/python3.10/site-packages/distilabel/pipeline/base.py#1046\1046]8;;\
                             stages:                                                                               
                              * Legend: 🚰 GeneratorStep 🌐 GlobalStep 🔄 Step                                     
                              * Stage 0:                                                                           
                                - 🚰 'load_data_from_dicts_0'                                                      
                                - 🔄 'text_generation_0'                                                           
                                - 🔄 'merge_columns_0'                                                             
                                - 🔄 's_n_s_conversation_feedback_0'                                               
                                - 🔄 'format_text_generation_d_p_o_0'                                              

                    INFO     ['distilabel.pipeline'] ⏳ Waiting for all the steps of stage 0 to        ]8;id=630930;file:///data/ephemeral/pro-nlp-finalproject-nlp-13/.venv/lib/python3.10/site-packages/distilabel/pipeline/base.py\base.py]8;;\:]8;id=259973;file:///data/ephemeral/pro-nlp-finalproject-nlp-13/.venv/lib/python3.10/site-packages/distilabel/pipeline/base.py#1382\1382]8;;\
                             load...                                                                               

[02/02/26 08:40:10] INFO     ['distilabel.pipeline'] ⏳ Steps from stage 0 loaded: 5/5                 ]8;id=357412;file:///data/ephemeral/pro-nlp-finalproject-nlp-13/.venv/lib/python3.10/site-packages/distilabel/pipeline/base.py\base.py]8;;\:]8;id=958785;file:///data/ephemeral/pro-nlp-finalproject-nlp-13/.venv/lib/python3.10/site-packages/distilabel/pipeline/base.py#1418\1418]8;;\
                              * 'load_data_from_dicts_0' replicas: 1/1                                             
                              * 'text_generation_0' replicas: 1/1                                                  
                              * 'merge_columns_0' replicas: 1/1                                                    
                              * 's_n_s_conversation_feedback_0' replicas: 1/1                                      
                              * 'format_text_generation_d_p_o_0' replicas: 1/1                                     

                    INFO     ['distilabel.pipeline'] ✅ All the steps from stage 0 have been loaded!   ]8;id=99411;file:///data/ephemeral/pro-nlp-finalproject-nlp-13/.venv/lib/python3.10/site-packages/distilabel/pipeline/base.py\base.py]8;;\:]8;id=282743;file:///data/ephemeral/pro-nlp-finalproject-nlp-13/.venv/lib/python3.10/site-packages/distilabel/pipeline/base.py#1422\1422]8;;\

...

                    INFO     ['distilabel.step.s_n_s_conversation_feedback_0'] 🏁 Finished      ]8;id=413722;file:///data/ephemeral/pro-nlp-finalproject-nlp-13/.venv/lib/python3.10/site-packages/distilabel/pipeline/step_wrapper.py\step_wrapper.py]8;;\:]8;id=48741;file:///data/ephemeral/pro-nlp-finalproject-nlp-13/.venv/lib/python3.10/site-packages/distilabel/pipeline/step_wrapper.py#129\129]8;;\
                             running step 's_n_s_conversation_feedback_0' (replica ID: 0)                          

                    INFO     ['distilabel.step.format_text_generation_d_p_o_0'] 📦 Processing   ]8;id=940238;file:///data/ephemeral/pro-nlp-finalproject-nlp-13/.venv/lib/python3.10/site-packages/distilabel/pipeline/step_wrapper.py\step_wrapper.py]8;;\:]8;id=799409;file:///data/ephemeral/pro-nlp-finalproject-nlp-13/.venv/lib/python3.10/site-packages/distilabel/pipeline/step_wrapper.py#230\230]8;;\
                             batch 1 in 'format_text_generation_d_p_o_0' (replica ID: 0)                           

                    INFO     ['distilabel.step.format_text_generation_d_p_o_0'] 📨 Step         ]8;id=263072;file:///data/ephemeral/pro-nlp-finalproject-nlp-13/.venv/lib/python3.10/site-packages/distilabel/pipeline/step_wrapper.py\step_wrapper.py]8;;\:]8;id=208567;file:///data/ephemeral/pro-nlp-finalproject-nlp-13/.venv/lib/python3.10/site-packages/distilabel/pipeline/step_wrapper.py#290\290]8;;\
                             'format_text_generation_d_p_o_0' sending batch 1 to output queue                      

                    INFO     ['distilabel.step.format_text_generation_d_p_o_0'] 🏁 Finished     ]8;id=447764;file:///data/ephemeral/pro-nlp-finalproject-nlp-13/.venv/lib/python3.10/site-packages/distilabel/pipeline/step_wrapper.py\step_wrapper.py]8;;\:]8;id=790772;file:///data/ephemeral/pro-nlp-finalproject-nlp-13/.venv/lib/python3.10/site-packages/distilabel/pipeline/step_wrapper.py#129\129]8;;\
                             running step 'format_text_generation_d_p_o_0' (replica ID: 0)                         

Generating train split: 0 examples [00:00, ? examples/s]

In [8]:
df = distiset["default"]["train"].to_pandas()

In [9]:
nan_ratings_df = df[df['ratings'].apply(
    lambda x: (not isinstance(x, (list, np.ndarray))) or  # 리스트 형태가 아님
              (len(x) != 2) or                           # 길이가 2가 아님 (보내주신 데이터 중 3개, 6개인 경우)
              (pd.isna(x).any())                         # 내부에 null/NaN이 포함됨
)]

# 2. 결과 확인
print(f"비정상 데이터 개수: {len(nan_ratings_df)}건")
display(nan_ratings_df)

비정상 데이터 개수: 5건


,instruction,distilabel_metadata,model_name,generations,ratings,rationales,prompt,prompt_id,chosen,chosen_rating,rejected,rejected_rating
39,그럼 우리도 서둘러 예약하자! 여행 가는 거 또 기대돼!,{'raw_input_s_n_s_conversation_feedback_0': [{...,solar-pro2,"[마좌 여행 갈 때마다 너무 재밌어! 이번에도 즐거운 추억 만들자!, 네, 서둘러 ...",[5],[구어체와 이모티콘 사용으로 자연스러우며 감정 표현이 풍부하고 참여 유도\nRati...,그럼 우리도 서둘러 예약하자! 여행 가는 거 또 기대돼!,5c665d1c5d1402e6b40d51af4c6ec1092ec453df3dd8d0...,[{'content': '그럼 우리도 서둘러 예약하자! 여행 가는 거 또 기대돼!'...,5,[{'content': '그럼 우리도 서둘러 예약하자! 여행 가는 거 또 기대돼!'...,5
43,렬루? 코로나 이후 첫 대면 행사래!,{'raw_input_s_n_s_conversation_feedback_0': [{...,solar-pro2,"[그래서 관광교류 활성화 방안을 모색한다는 거지?, 드디어 포스트 코로나 시대로의 ...",[4],[구어체와 질문을 포함해 자연스러우나 맥락 연결이 다소 부족합니다.\nRating:...,렬루? 코로나 이후 첫 대면 행사래!,b3665bf11cbf6ed5916eb6abb2dfb486a3cc64addca769...,"[{'content': '렬루? 코로나 이후 첫 대면 행사래!', 'role': '...",4,"[{'content': '렬루? 코로나 이후 첫 대면 행사래!', 'role': '...",4
71,나중에 가서 구경해야겠다,{'raw_input_s_n_s_conversation_feedback_0': [{...,solar-pro2,"[나도 같이 가자! 같이 산책하면 좋을 것 같아, 차후에 시간을 내어 방문해보시는 ...",[4],[구어체와 공감을 담은 참여 유도로 자연스러운 대화체 특성을 보인다. \nRati...,나중에 가서 구경해야겠다,bf90a3b50cd6449f14a2744e779b21318f96d9a5c59cc6...,"[{'content': '나중에 가서 구경해야겠다', 'role': 'user'},...",4,"[{'content': '나중에 가서 구경해야겠다', 'role': 'user'},...",4
87,그래서 MZ세대를 겨냥한 콘텐츠도 준비했대.,{'raw_input_s_n_s_conversation_feedback_0': [{...,solar-pro2,"[MZ세대들은 한류에 관심 많을 텐데 좋은 아이디어네!, 최근 기업의 마케팅 전략에...",[5],[구어체 표현과 이모티콘을 사용해 자연스러우며 맥락 이해가 명확하고 간결하다. \...,그래서 MZ세대를 겨냥한 콘텐츠도 준비했대.,4572a6241e4bfedf67deb837523455fb8a9ff4090c2424...,"[{'content': '그래서 MZ세대를 겨냥한 콘텐츠도 준비했대.', 'role...",5,"[{'content': '그래서 MZ세대를 겨냥한 콘텐츠도 준비했대.', 'role...",5
94,다른 곳에서도 겨울축제가 열리는 거 알아?,{'raw_input_s_n_s_conversation_feedback_0': [{...,solar-pro2,"[응! 인제군에서는 내년 2월에 빙어축제가 열린대!, 국내 여러 지역에서도 겨울축제...",[5],"[구어체, 이모티콘, 줄임말로 자연스러우며, 맥락을 이해하고 간결하게 참여 유도 ...",다른 곳에서도 겨울축제가 열리는 거 알아?,002a41ea754a9e41884ff6aa6433586c50185c7825af0a...,"[{'content': '다른 곳에서도 겨울축제가 열리는 거 알아?', 'role'...",5,"[{'content': '다른 곳에서도 겨울축제가 열리는 거 알아?', 'role'...",5


In [100]:
raw_text_input = df['distilabel_metadata'].iloc[98]['raw_input_s_n_s_conversation_feedback_0']
print("--- 판사의 반별을 위한 input ---")
print(raw_text_input)

--- 판사의 반별을 위한 input ---
[{'content': '당신은 실제 SNS 대화체(카카오톡, DM 등)를 판별하는 전문 평가가입니다.\n\n당신의 출력은 오직 아래에 명시된 [출력 형식]만 허용됩니다.\n출력 형식을 벗어나는 경우, 해당 응답은 파싱 오류로 처리됩니다.\n\n[절대 금지 사항]\n- 추가 설명, 요약, 평가 근거, 분석, <think> 태그, 요류 설명\n- 구분선(---), 제목, 번호 매기기\n- 출력 전후의 공백 줄 또는 문장\n\n[반드시 준수해야 할 작성 규칙]\n1. 반드시 아래의 [출력 형식]만 사용하십시오. 강하게 강제합니다.\n2. Rating과 Rationale 외의 텍스트는 절대 출력하지 마십시오.\n3. 각 응답 묶음(Rating + Rationale) 사이에는 정확히 빈 줄 한 줄(줄바꿈 두 번)만 허용되며, 마지막 Rationale 줄 뒤에는 추가 줄바꿈이나 텍스트 없이 즉시 종료하십시오.\n            \n***[출력 형식]***\nRating: [첫 번째 응답의 1-5점 사이의 점수]\nRationale: [첫 번째 응답에 대한 간결한 이유 한 문장]\n\nRating: [두 번째 응답의 1-5점 사이의 점수]\nRationale: [두 번째 응답에 대한 간결한 이유 한 문장]\n', 'role': 'system'}
 {'content': '당신은 입력된 텍스트가 실제 SNS(카카오톡, DM 등) 대화체인지 판별해야 합니다.\n아래의 다차원 평가 기준을 종합적으로 고려하여 점수를 부여하십시오.\n\n[종합 평가 기준 및 가중치]\n- 자연스러움 (0.25): 실제 SNS에서 쓰이는 구어체, 이모티콘, 줄임말 사용 정도\n- 맥락 적합성 (0.25): 이전 대화 맥락을 이해하고 자연스럽게 이어지는가\n- 참여 유도 (0.20): 공감, 질문 등으로 대화를 이어가려는 요소가 있는가\n- 감정 적절성 (0.15): 상황에 맞는 톤과 감정, 공감 표현이 있는가\n- 간결성 (0.15): SNS 특성에

In [11]:
raw_text_output = df['distilabel_metadata'].iloc[43]['raw_output_s_n_s_conversation_feedback_0']
print("--- 판사의 반별을 위한 output ---")
print(raw_text_output)

--- 판사의 반별을 위한 output ---
Rating: 4
Rationale: 구어체와 질문을 포함해 자연스러우나 맥락 연결이 다소 부족합니다.
Rating: 2
Rationale: 설명조 문장과 형식적 어조로 SNS 대화체 특성과 맞지 않습니다.


In [ ]:
# 결과를 담을 리스트
dpo_rows = []

for idx, row in df.iterrows():
    ratings = row['ratings']
    generations = row['generations']
    
    # 1. ratings가 array([float, float]) 형태인지 엄격하게 확인
    # 리스트나 넘파이 배열이 아니거나, 길이가 2가 아니거나, 내부 요소에 NaN이 있으면 pass
    if not isinstance(ratings, (list, np.ndarray)) or len(ratings) != 2:
        continue
    if pd.isna(ratings).any():
        continue

    # 2. 값 비교 및 chosen/rejected 할당
    # ratings[0]: 첫 번째 답변 점수, ratings[1]: 두 번째 답변 점수
    try:
        if ratings[0] >= ratings[1]:
            chosen = generations[0]
            rejected = generations[1]
        else:
            chosen = generations[1]
            rejected = generations[0]
            
        # 3. 새로운 데이터 구조 생성
        new_row = row.to_dict()
        new_row['chosen'] = chosen
        new_row['rejected'] = rejected
        dpo_rows.append(new_row)
        
    except (IndexError, TypeError):
        # generations 리스트에 값이 부족하거나 다른 에러 발생 시 건너뜀
        continue

In [42]:
dpo_df = pd.DataFrame(dpo_rows)

print(f"변환 완료: {len(dpo_df)}개 행 (원본: {len(df)}개)")
display(dpo_df[['instruction', 'chosen', 'rejected', 'ratings']].head())

변환 완료: 93개 행 (원본: 100개)


,instruction,chosen,rejected,ratings
0,여름 휴가철 어디로 갈까?,나는 해변으로 가고 싶어!,여름 휴가 시즌이 다가오고 있습니다. 올해는 국내외 여러 여행지 중 어디로 떠나면 ...,"[5.0, 1.0]"
1,경기도 청소년 관악제의 목적은 뭐임?,경기도 청소년 관악제는 청소년들에게 예술적 재능을 발휘할 수 있는 기회를 제공하고 ...,"경기도 청소년 관악제는 경기도 내 청소년들의 문화예술 창작활동을 장려하고, 그들이 ...","[3.0, 1.0]"
2,거창한 책보다는 가벼운 소설이나 만화책을 읽어보는 건 ㅇㄸ?,그게 좋을 것 같다. 그래서 쉬는 시간에 책을 읽어야지,가벼운 소설이나 만화책도 나름의 가치와 재미를 지니고 있습니다. 하지만 때로는 좀 ...,"[3.0, 1.0]"
3,렬루? 그럼 어떤 건축양식이 있는 거야?,"고딕, 르네상스, 바로크 등이 있어! 그리고 1920년에는 합스부르크 왕가가 여름 ...","건축 양식에는 다양한 종류가 있습니다. 고대부터 현대까지, 그리고 지역과 문화에 따...","[3.0, 1.0]"
4,"공항 면세점 상품 구성도 새로 하고, 할인 이벤트도 준비하고 있다고 하더라고.",그렇구나. 잘 준비해서 좋은 결과 있었으면 해.,최근 공항 면세점의 개편 소식이 전해졌습니다. 이들은 단순한 상품 구성의 변경을 넘...,"[4.0, 1.0]"


In [ ]:
df[['instruction', 'chosen', 'rejected', 'ratings']].to_json("sns_dpo_data_raw.json", orient="records", indent=4, force_ascii=False)

In [44]:
dpo_df[['instruction', 'chosen', 'rejected', 'ratings']].to_json("sns_dpo_data.json", orient="records", indent=4, force_ascii=False)